In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils

In [ ]:
parallelism.set_max_num_tbb_threads(8)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import periodic_unit_helper

In [ ]:
import importlib

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
m, fuseMarkers, brdyWallMarkers = periodic_unit_helper.get_mesh_input()

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) != 0)

In [ ]:
ipu.sheet.pressure = 10

In [ ]:
isheet.pressure = 10

In [ ]:
# sA = ipu.getPeriodicPatchToInflatableSheetMapTranspose()

# sA_triplet = sA.getTripletMatrix()

# (sA_triplet.m, sA_triplet.n)

# nA = np.zeros((sA_triplet.m , sA_triplet.n))

# for entry in sA_triplet.entries():
#     nA[entry.i, entry.j] = entry.v

In [ ]:
# m.vertices()

In [ ]:
from tri_mesh_viewer import TriMeshViewer
sheet_viewer = TriMeshViewer(isheet, width=768, height=640)
sheet_viewer.showWireframe(True)
sheet_viewer.show()

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
import time, vis
benchmark.reset()
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
isheet.pressure = 20 * 3.75
opts.niter = 10
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update()
cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars[:3], opts, callback=cb)
benchmark.report()

In [ ]:
import periodic_unit_helper

In [ ]:
fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

In [ ]:
import time, vis
benchmark.reset()
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.pressure = 20 * 3.75
opts.niter = 10
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update()
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb)
benchmark.report()

### Apply perturbation and check that the sheet is deformed with periodic condition

#### IPU

In [ ]:
fd_perturb = np.random.uniform(-1e-2, 1e-2, ipu.numVars())
fd_perturb[:3] *= 0

In [ ]:
ipu.setVars(ipu.getVars() + fd_perturb)

In [ ]:
viewer.update()

In [ ]:
import fd_validation

In [ ]:
ipu.energy()

#### Inflatable Sheet

In [ ]:
fd_perturb = np.random.uniform(-1e-2, 1e-2, isheet.numVars())

In [ ]:
isheet.setVars(isheet.getVars() + fd_perturb)

In [ ]:
sheet_viewer.update()

### Finite difference validation

In [ ]:
Pressure = inflation.InflatableSheet.EnergyType.Pressure
Elastic  = inflation.InflatableSheet.EnergyType.Elastic
Full     = inflation.InflatableSheet.EnergyType.Full

In [ ]:
ipu.sheet.pressure = 10

In [ ]:
fd_validation.gradConvergencePlot(ipu, customArgs = {"energyType": Elastic})

In [ ]:
fd_validation.gradConvergencePlot(ipu, customArgs = {"energyType": Pressure})

In [ ]:
isheet.pressure = 10

In [ ]:
fd_validation.gradConvergencePlot(isheet, customArgs = {"energyType": Elastic})

In [ ]:
fd_validation.gradConvergencePlot(isheet, customArgs = {"energyType": Pressure})

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": Elastic})

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": Pressure})

In [ ]:
fd_validation.hessConvergencePlot(isheet, customArgs = {"energyType": Elastic})

In [ ]:
fd_validation.hessConvergencePlot(isheet, customArgs = {"energyType": Pressure})

In [ ]:
isheet.setUseTensionFieldEnergy(False)



In [ ]:
fd_validation.hessConvergencePlot(isheet, customArgs = {"energyType": inflation.InflatableSheet.EnergyType.Pressure})